# 04 · Release candidate 패키지 · 회귀 검사 · 승격 절차

02 에서 `status = candidate` 가 나온 경우에만 의미가 있습니다. 이 노트북은 **배포하거나 active 를 바꾸지 않습니다.** 다음을 수행합니다.
1. 후보 패키지 확인: ONNX 해시 · manifest(feature 순서·전처리·opset·test vectors·지원 범위·한계) · 모델 카드
2. Python ↔ ONNX Runtime parity 재확인(허용 오차 abs 1e-5 · rel 1e-4), 크기·파라미터 수 예산
3. 브라우저 측 테스트와 같은 검증(해시 불일치·스키마·NaN → fail-closed)이 저장소 테스트로 존재함을 확인
4. registry 패치 파일 생성 → 개발자가 **검토된 commit** 으로 `models/gaze/<id>/` 와 `model-registry.json(candidates)` 를 올림
5. 승격은 사람이 한다: shadow(기록만) → 승인된 소규모 canary → active. 각 단계는 registry history 에 승인자·시각·이전 active·rollback 대상과 함께 기록

In [ ]:
#@title ① 환경
import os, subprocess, sys, json, glob
if not os.path.isdir('/content/neurolens'): subprocess.check_call(['git', 'clone', '--depth', '50', 'https://github.com/jeebs0627/neurolens.git', '/content/neurolens'])
os.chdir('/content/neurolens'); sys.path.insert(0, '/content/neurolens/tools/colab')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'onnx', 'onnxruntime'])
from google.colab import drive  # type: ignore
drive.mount('/content/drive'); WORK = '/content/drive/MyDrive/neurolens-training/work'

In [ ]:
#@title ② 후보 확인 · parity 재검사 · 예산
import numpy as np, onnxruntime as ort
from nlcolab.canonical import file_sha256
from nlcolab.safe_io import load_json
cands = sorted(glob.glob(f'{WORK}/gaze/candidate/models/gaze/*/manifest.json'))
if not cands: print('candidate 없음 — 02 의 상태를 확인하세요 (insufficient_data / evaluated 이면 승격 대상이 없습니다)')
for m in cands:
    man = load_json(m); d = os.path.dirname(m); onnx_path = f'{d}/model.onnx'
    sha = file_sha256(onnx_path); print(man['id'], man['version'], '해시 일치' if sha == man['sha256'] else '⚠ 해시 불일치', f"{os.path.getsize(onnx_path)} bytes", 'opset', man['opset'])
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    tol = man['tolerance']; ok = True
    for tv in man['testVectors']:
        y = sess.run(None, {man['input']['name']: np.array([tv['input']], dtype=np.float32)})[0][0]
        ok &= all(abs(y[i] - tv['output'][i]) <= tol['abs'] + tol['rel'] * abs(tv['output'][i]) for i in range(2))
    print('  test vectors parity:', ok, '· features:', man['features'], '· contract:', man['contract'], '· units:', man['output']['units'])
    print('  release state:', man['release'], '\n  limitations:', *[f'\n   - {l}' for l in man['limitations']])

In [ ]:
#@title ③ registry 패치와 승격 체크리스트 출력
patch = f'{WORK}/gaze/model-registry.candidate.json'
if os.path.exists(patch):
    print(json.dumps(load_json(patch), ensure_ascii=False, indent=2)[:4000])
print('''
승격 체크리스트 (사람이 수행 · COLAB_TRAINING_RUNBOOK.md 참조)
 [ ] 검토된 commit 으로 models/gaze/<id>/ + model-registry.json(candidates) 반영 → 배포
 [ ] registry.shadow 에 후보 지정 → /condition 실기기에서 shadow 지연·오류·출력 분포 확인(gazeModel.latency/guardReasons 가 연구 기록에 남음)
 [ ] 저사양/모바일 1종 이상 실측 p95 ≤ 예산(5~8ms) · WASM 로드 · CSP/CORS/캐시 확인
 [ ] 승인된 소규모 canary(사용자 동의 범위 안) → 정확도·coverage 회귀 없음
 [ ] registry.active 지정 + history 에 approvedBy/approvedAt/previousActive/rollbackTo 기록 (세션 중 교체 없음 · 다음 검사부터)
 [ ] rollback 절차 리허설: active=null 또는 이전 항목으로 되돌리는 commit → 배포
''')